In [19]:
import pandas as pd
import plotly.express as px

# Load and prepare data
xls = pd.ExcelFile('merged.xlsx')
df_merged = pd.read_excel(xls, xls.sheet_names[0])

df_merged['COGS'] = (
    df_merged['HARDWARE']
    + df_merged['SOFTWARE']
    + df_merged['MANPOWER']
)

df_merged['GROSS_PROFIT'] = df_merged['REVENUE'] - df_merged['COGS']
df_merged['GROSS_MARGIN'] = (
    df_merged['GROSS_PROFIT'] / df_merged['REVENUE']
) * 100

# 2D Scatter Plot
fig = px.scatter(
    df_merged,
    x='REVENUE',
    y='GROSS_PROFIT',
    color='TYPE',
    size='MANPOWER',
    hover_name='Client ID',
    hover_data={
        'SECTOR': True,
        'COUNTRY': True,
        'STAFF STRENGTH': True,
        'NPS RATING': True,
        'GROSS_MARGIN': ':.2f',
        'REVENUE': ':,.2f',
        'MANPOWER': ':,.2f',
        'GROSS_PROFIT': ':,.2f'
    },
    title='<b>Revenue vs Gross Profit by Client Type</b>',
    labels={
        'REVENUE': 'Revenue ($)',
        'GROSS_PROFIT': 'Gross Profit ($)',
        'TYPE': 'Client Type',
        'MANPOWER': 'Manpower Cost ($)'
    },
    color_discrete_sequence=px.colors.qualitative.Bold
)

fig.update_layout(
    template='plotly_white',
    width=900,
    height=600
)

fig.show()

In [15]:
import plotly.graph_objects as go

# Group data by Client Type to analyze cost components
cost_df = df_merged.groupby('TYPE', as_index=False).agg({
    'HARDWARE': 'sum',
    'SOFTWARE': 'sum',
    'MANPOWER': 'sum'
})

# Create Stacked Bar Chart using Graph Objects
fig_go1 = go.Figure(data=[
    go.Bar(name='Hardware Costs', x=cost_df['TYPE'], y=cost_df['HARDWARE'], marker_color='#636EFA'),
    go.Bar(name='Software Costs', x=cost_df['TYPE'], y=cost_df['SOFTWARE'], marker_color='#EF553B'),
    go.Bar(name='Manpower Costs', x=cost_df['TYPE'], y=cost_df['MANPOWER'], marker_color='#00CC96')
])

# Change the bar mode to stacked and add interactive buttons/dropdowns
fig_go1.update_layout(
    barmode='stack',
    title='<b>Cost Driver Breakdown (Hardware vs Software vs Manpower) by Client Type</b>',
    xaxis_title='Client Type',
    yaxis_title='Cost ($)',
    template='plotly_white',
    legend=dict(title="Cost Components"),
    updatemenus=[
        dict(
            type="buttons",
            direction="left",
            x=0.5,
            y=1.15,
            xanchor="center",
            buttons=list([
                dict(label="Stacked Bar", method="relayout", args=[{"barmode": "stack"}]),
                dict(label="Grouped Bar", method="relayout", args=[{"barmode": "group"}])
            ])
        )
    ]
)

fig_go1.show()

In [12]:
# Group data by Client Type to get average scores for satisfaction dimensions and NPS
satisfaction_df = df_merged.groupby('TYPE', as_index=False).agg({
    'PRESALES AND PARTNERSHIP': 'mean',
    'TECHNICAL EXPERTISE': 'mean',
    'PROJECT DELIVERY': 'mean',
    'POST-SALES SUPPORT': 'mean',
    'NPS RATING': 'mean'
})

# Create a Radar/Polar Chart or Grouped Bar Chart comparing service drivers
categories = ['Presales & Partnership', 'Technical Expertise', 'Project Delivery', 'Post-Sales Support', 'NPS Rating']

fig_go2 = go.Figure()

for i, row in satisfaction_df.iterrows():
    fig_go2.add_trace(go.Scatterpolar(
        r=[
            row['PRESALES AND PARTNERSHIP'], 
            row['TECHNICAL EXPERTISE'], 
            row['PROJECT DELIVERY'], 
            row['POST-SALES SUPPORT'], 
            row['NPS RATING']
        ],
        theta=categories,
        fill='toself',
        name=str(row['TYPE'])
    ))

fig_go2.update_layout(
    polar=dict(
        radialaxis=dict(
            visible=True,
            range=[0, df_merged[['PRESALES AND PARTNERSHIP', 'TECHNICAL EXPERTISE', 'PROJECT DELIVERY', 'POST-SALES SUPPORT', 'NPS RATING']].max().max()]
        )),
    title='<b>Customer Satisfaction Dimensions & NPS Profile across Client Types</b>',
    template='plotly_white'
)

fig_go2.show()